# Code Documentation Assistant — dev branch
## LangGraph Pipeline Walkthrough & Graph Visualisation

This notebook:
1. Builds the LangGraph `StateGraph` and renders it visually
2. Walks through each node with mock inputs (no Ollama required for the graph itself)
3. Demonstrates the HITL checkpoint flow
4. Shows MLflow logging integration

Run with: `jupyter lab notebooks/pipeline_walkthrough.ipynb`

In [ ]:
import sys, os
sys.path.insert(0, '../src')

# Set env before importing agent modules
os.environ.update({
    'OLLAMA_HOST': 'http://localhost:11434',
    'CHROMA_HOST': 'http://localhost:8000',
    'MLFLOW_TRACKING_URI': 'http://localhost:5000',
    'HITL_ENABLED': 'true',
    'CONFIDENCE_THRESHOLD': '0.45',
    'MAX_RETRIEVAL_ATTEMPTS': '3',
})

from agent_graph import build_graph, get_graph_mermaid
from agent_state import AgentState, ToolCall, Chunk
print('Imports OK')

## 1. Graph topology

`draw_mermaid_png()` requires network access to mermaid.ink.  
If that's blocked, `draw_mermaid()` returns the Mermaid source which renders below via the `%%mermaid` cell magic (requires the `mermaid-js` Jupyter extension), or paste the source into https://mermaid.live.

In [ ]:
graph = build_graph()

# Try PNG first (needs mermaid.ink access)
try:
    from IPython.display import Image
    png_bytes = graph.get_graph().draw_mermaid_png()
    display(Image(png_bytes))
    print('Graph rendered as PNG')
except Exception as e:
    print(f'PNG render unavailable ({e}). Mermaid source below:')
    mermaid_src = graph.get_graph().draw_mermaid()
    print(mermaid_src)
    # Also save for external rendering
    with open('/tmp/agent_graph.mmd', 'w') as f:
        f.write(mermaid_src)
    print('\nSaved to /tmp/agent_graph.mmd — paste at https://mermaid.live')

## 2. Inspect node list and edge structure

In [ ]:
g = graph.get_graph()
print('Nodes:')
for node in g.nodes:
    print(f'  {node}')

print('\nEdges:')
for edge in g.edges:
    source, target = edge[0], edge[1]
    conditional = len(edge) > 2 and edge[2]
    marker = ' (conditional)' if conditional else ''
    print(f'  {source} → {target}{marker}')

## 3. Run the pipeline (HITL disabled for notebook demo)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

os.environ['HITL_ENABLED'] = 'false'  # auto-approve for notebook
graph_no_hitl = build_graph(checkpointer=MemorySaver())

# Use a local repo path — change to an actual repo you have
REPO_PATH = os.environ.get('REPO_PATH', '/tmp/test-repo')

# Create a minimal test repo if it doesn't exist
import subprocess
os.makedirs(REPO_PATH, exist_ok=True)
with open(f'{REPO_PATH}/example.py', 'w') as f:
    f.write('''
def calculate_fibonacci(n: int) -> int:
    """Return the nth Fibonacci number."""
    if n <= 1:
        return n
    return calculate_fibonacci(n - 1) + calculate_fibonacci(n - 2)

class DataProcessor:
    """Processes raw data records."""
    def __init__(self, config: dict):
        self.config = config
        self._cache = {}

    def process(self, record: dict) -> dict:
        """Apply configured transformations to a single record."""
        result = {}
        for key, transform in self.config.items():
            result[key] = transform(record.get(key))
        return result
''')
print(f'Test repo at {REPO_PATH}')

In [ ]:
initial_state = {
    'query': 'What does the DataProcessor class do and how is it initialised?',
    'repo_path': REPO_PATH,
    'proposed_tool_calls': [],
    'hitl_checkpoint': None,
    'approved_tool_calls': [],
    'executed_tool_calls': [],
    'retrieved_chunks': [],
    'confidence_scores': [],
    'retrieval_attempts': 0,
    'max_retrieval_attempts': 3,
    'supervisor_adjustments': [],
    'proceed_to_generation': False,
    'final_context': '',
    'response': '',
    'source_attribution': [],
    'mlflow_run_id': None,
    'total_latency_ms': None,
}

config = {'configurable': {'thread_id': 'notebook-demo-1'}}

# Stream node-by-node to see what's happening
print('Streaming graph execution:\n')
for step in graph_no_hitl.stream(initial_state, config=config, stream_mode='updates'):
    node_name = list(step.keys())[0]
    update = step[node_name]
    print(f'▶ Node: {node_name}')
    if 'proposed_tool_calls' in update:
        for tc in update['proposed_tool_calls']:
            print(f'  Tool: {tc.tool_name} | Args: {tc.args}')
    if 'retrieved_chunks' in update:
        print(f'  Chunks retrieved: {len(update["retrieved_chunks"])}')
    if 'supervisor_adjustments' in update:
        for adj in update['supervisor_adjustments']:
            print(f'  Supervisor: {adj.action} — {adj.reason}')
    if 'response' in update and update['response']:
        print(f'  Response (first 200 chars): {update["response"][:200]}...')
    print()

## 4. HITL flow demonstration

With `HITL_ENABLED=true`, the graph pauses at `hitl_checkpoint`.  
Resume with `graph.invoke(Command(resume={...}), config=config)`.

In [ ]:
from langgraph.types import Command

os.environ['HITL_ENABLED'] = 'true'
graph_hitl = build_graph(checkpointer=MemorySaver())
config_hitl = {'configurable': {'thread_id': 'notebook-hitl-demo'}}

# Initial invoke — will pause at hitl_checkpoint
initial_state['max_retrieval_attempts'] = 2  # faster for demo

try:
    result = graph_hitl.invoke(initial_state, config=config_hitl)
except Exception as e:
    print(f'Graph paused (expected): {type(e).__name__}')

# Check what the graph is waiting on
snapshot = graph_hitl.get_state(config_hitl)
print('Graph is paused at:', snapshot.next)
print('Proposed tool calls:')
for tc in snapshot.values.get('proposed_tool_calls', []):
    print(f'  {tc.tool_name}: {tc.args}')

In [ ]:
# Human approves — resume the graph
human_response = {
    'decision': 'approved',
    'tool_calls': [
        {'tool_name': tc.tool_name, 'args': tc.args}
        for tc in snapshot.values.get('proposed_tool_calls', [])
    ],
    'feedback': 'Looks good',
}

final_state = graph_hitl.invoke(Command(resume=human_response), config=config_hitl)
print('Response:', final_state.get('response', '[no response]')[:300])
print('\nSources:', final_state.get('source_attribution', []))
print('Supervisor adjustments:', len(final_state.get('supervisor_adjustments', [])))